# VietHandOCR Part 2: Baseline Evaluation

Welcome to **Part 2** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 1: Data Preparation & EDA](./01_Data_Preparation_and_EDA.ipynb)
- **Next Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)

## Introduction
This notebook establishes the baseline performance of the standard `vgg_transformer` model on the writer-independent dataset splits. By measuring the Zero-shot performance (without any prior task-specific fine-tuning or image processing), we create a benchmark to evaluate the quantitative impact of the upcoming Digital Image Processing (DIP) heuristics and domain-adaptation fine-tuning.

## Objectives
1. **Inference**: Perform Zero-shot inference across all text levels (word, line, paragraph).
2. **Metrics Calculation**: Compute Character Error Rate (CER), Word Error Rate (WER), Exact Match accuracy, and BLEU-2 scores.
3. **Error Analysis Logging**: Export detailed step-by-step evaluation logs and aggregated metrics for further analysis.


In [1]:
!pip install -q vietocr jiwer nltk 


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
# HOTFIX: Prevent Kaggle Pillow memory corruption error during Save & Run All
import PIL._util
import os
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir

import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

def calculate_metrics(predictions, targets):
    '''Calculates CER, WER, Exact Match, and BLEU.'''
    cer = jiwer.cer(targets, predictions)
    wer = jiwer.wer(targets, predictions)
    exact_match = sum(1 for p, t in zip(predictions, targets) if p == t) / len(targets)
    
    bleu_scores = []
    for p, t in zip(predictions, targets):
        reference = [nltk.word_tokenize(t.lower())]
        candidate = nltk.word_tokenize(p.lower())
        if len(candidate) == 0:
            bleu_scores.append(0.0)
            continue
        try:
            bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, weights=(0.5, 0.5))
            bleu_scores.append(bleu)
        except Exception:
            bleu_scores.append(0.0)
            
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    
    return {"CER": cer, "WER": wer, "Exact Match": exact_match, "BLEU": avg_bleu}


In [3]:
def evaluate_baseline(predictor, test_txt_path, images_base_dir, level_name='all'):
    with open(test_txt_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    targets = []
    predictions = []
    results_detail = []
    
    print(f"  [1/3] READING INPUT DATA...")
    print(f"    - File input: {test_txt_path}")
    print(f"    - Base Image Dir: {images_base_dir}")
    print(f"    - Total data records (images): {len(lines)}")
    print(f"  [2/3] STARTING ZERO-SHOT INFERENCE...")
    for line in tqdm(lines):
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) != 2: continue
            
        rel_img_path, ground_truth = parts
        ground_truth = ground_truth.strip()
        if not ground_truth: continue # Skip empty ground truth
        
        full_img_path = os.path.join(images_base_dir, rel_img_path)
        
        try:
            with Image.open(full_img_path) as img:
                pred = predictor.predict(img).strip()
            
            targets.append(ground_truth)
            predictions.append(pred)
            
            results_detail.append({
                'image_path': rel_img_path,
                'ground_truth': ground_truth,
                'prediction': pred,
                'is_exact_match': ground_truth == pred
            })
        except Exception as e:
            print(f"Error processing {full_img_path}: {e}")
            
    if not targets:
        print("No valid targets found. Skipping metrics calculation.")
        return {}, pd.DataFrame()
        
    metrics = calculate_metrics(predictions, targets)
    print("\n" + "="*40)
    print(f"🏆 BASELINE RESULTS (Zero-shot) - LEVEL: {level_name.upper()}")
    print("="*40)
    for k, v in metrics.items():
        print(f"{k:<15}: {v:.4f}")
    print("="*40)
    
    df_results = pd.DataFrame(results_detail)
    df_results.to_csv(f'baseline_error_analysis_{level_name}.csv', index=False, encoding='utf-8')
    output_csv = f'baseline_error_analysis_{level_name}.csv'
    print(f"  [3/3] EXPORTING EVALUATION RESULTS...")
    print(f"    - Detailed error analysis output path: {os.path.join(os.getcwd(), output_csv)}")
    print(f"    - Successfully saved {len(df_results)} prediction records.")
    
    return metrics, df_results


In [4]:
import os
import glob
import gc
import torch
import pandas as pd
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# Robust Dataset Path Resolution
def get_dataset_path():
    kaggle_input = '/kaggle/input'
    local_input = 'VietHandOCR_Datasets'
    if os.path.exists(kaggle_input):
        # Look for the structure containing UIT_HWDB_word or UIT_HWDB_line
        for root, dirs, files in os.walk(kaggle_input):
            if any('UIT_HWDB_' in d for d in dirs):
                return root
        return kaggle_input
    return local_input

base_dataset_path = get_dataset_path()

# Locate all generated test_*.txt and val_*.txt evaluation sets in Kaggle /input or local directory
eval_files = {}
search_dirs = ['../input', '.']
for sdir in search_dirs:
    if os.path.exists(sdir):
        for root, dirs, files in os.walk(sdir):
            # PREVENT DEEP TRAVERSAL INTO IMAGE DIRECTORIES to avoid Kaggle NFS I/O bottlenecks
            dirs[:] = [d for d in dirs if not d.startswith('UIT_HWDB_') and d not in ['.git', '.ipynb_checkpoints']]
            for file in files:
                if (file.startswith('test_') or file.startswith('val_')) and file.endswith('.txt'):
                    # Extract split_level name, e.g. test_line.txt -> test_line
                    name = file.replace('.txt', '')
                    if name not in eval_files: # Prefer the first found path
                        eval_files[name] = os.path.join(root, file)

if not eval_files:
    print("Warning: No evaluation files (test_*.txt or val_*.txt) found. Please run 01_Data_Preparation_and_EDA.ipynb first to generate them.")
else:
    print(f"Found dataset at: {base_dataset_path}")
    print(f"Found {len(eval_files)} evaluation files: {list(eval_files.keys())}")
    
    print("\n" + "="*50)
    print("🤖 LOADING MODEL (vgg_transformer)")
    print("="*50)
    
    # Load the standard VGG19 + Transformer model for the baseline evaluation
    config = Cfg.load_config_from_name('vgg_transformer')
    
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    predictor = Predictor(config)
    print(f"Loaded predictor successfully on {config['device']}!")
    
    all_metrics = {}
    for name, txt_path in eval_files.items():
        print(f"\n{'='*50}")
        print(f"🚀 EVALUATING: {name.upper()}")
        print(f"{'='*50}")
        
        metrics, df_results = evaluate_baseline(predictor, txt_path, base_dataset_path, level_name=name)
        if metrics:
            all_metrics[name] = metrics
            
        # Free memory after each evaluation run to respect Kaggle constraints
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    print(f"\n{'*'*50}")
    print(f"📊 SUMMARY OF ALL EVALUATIONS (Zero-shot VGG19)")
    print(f"{'*'*50}")
    
    summary_records = []
    for name, metrics in all_metrics.items():
        print(f"--- Split: {name.upper()} ---")
        row = {'Split': name.upper()}
        for k, v in metrics.items():
            print(f"  {k:<12}: {v:.4f}")
            row[k] = v
        summary_records.append(row)
    print(f"{'*'*50}")
    
    # Export global summary CSV
    summary_df = pd.DataFrame(summary_records)
    summary_csv_path = 'baseline_summary_metrics.csv'
    summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
    print(f"\n✅ EXPORTED SUMMARY METRICS: {os.path.join(os.getcwd(), summary_csv_path)}")


Found dataset at: /kaggle/input/datasets/trnchihong/viethandocr-data
Found 8 evaluation files: ['val_all', 'test_line', 'test_word', 'test_paragraph', 'val_word', 'val_paragraph', 'val_line', 'test_all']

🤖 LOADING MODEL (vgg_transformer)
Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:03<00:00, 162MB/s]
18533it [00:13, 1416.27it/s]


Loaded predictor successfully on cuda:0!

🚀 EVALUATING: VAL_ALL
  [1/3] READING INPUT DATA...
    - File input: ../input/notebooks/trnchihong/01-data-preparation-and-eda/val_all.txt
    - Base Image Dir: /kaggle/input/datasets/trnchihong/viethandocr-data
    - Total data records (images): 11952
  [2/3] STARTING ZERO-SHOT INFERENCE...


  0%|          | 0/11952 [00:00<?, ?it/s]


🏆 BASELINE RESULTS (Zero-shot) - LEVEL: VAL_ALL
CER            : 0.5586
WER            : 0.7805
Exact Match    : 0.1984
BLEU           : 0.0215
  [3/3] EXPORTING EVALUATION RESULTS...
    - Detailed error analysis output path: /kaggle/working/baseline_error_analysis_val_all.csv
    - Successfully saved 11952 prediction records.

🚀 EVALUATING: TEST_LINE
  [1/3] READING INPUT DATA...
    - File input: ../input/notebooks/trnchihong/01-data-preparation-and-eda/test_line.txt
    - Base Image Dir: /kaggle/input/datasets/trnchihong/viethandocr-data
    - Total data records (images): 201
  [2/3] STARTING ZERO-SHOT INFERENCE...


...


🏆 BASELINE RESULTS (Zero-shot) - LEVEL: VAL_PARAGRAPH
CER            : 0.9861
WER            : 0.9986
Exact Match    : 0.0000
BLEU           : 0.0066
  [3/3] EXPORTING EVALUATION RESULTS...
    - Detailed error analysis output path: /kaggle/working/baseline_error_analysis_val_paragraph.csv
    - Successfully saved 121 prediction records.

🚀 EVALUATING: VAL_LINE
  [1/3] READING INPUT DATA...
    - File input: ../input/notebooks/trnchihong/01-data-preparation-and-eda/val_line.txt
    - Base Image Dir: /kaggle/input/datasets/trnchihong/viethandocr-data
    - Total data records (images): 678
  [2/3] STARTING ZERO-SHOT INFERENCE...


  0%|          | 0/678 [00:00<?, ?it/s]


🏆 BASELINE RESULTS (Zero-shot) - LEVEL: VAL_LINE
CER            : 0.1991
WER            : 0.5319
Exact Match    : 0.0000
BLEU           : 0.3778
  [3/3] EXPORTING EVALUATION RESULTS...
    - Detailed error analysis output path: /kaggle/working/baseline_error_analysis_val_line.csv
    - Successfully saved 678 prediction records.

🚀 EVALUATING: TEST_ALL
  [1/3] READING INPUT DATA...
    - File input: ../input/notebooks/trnchihong/01-data-preparation-and-eda/test_all.txt
    - Base Image Dir: /kaggle/input/datasets/trnchihong/viethandocr-data
    - Total data records (images): 3113
  [2/3] STARTING ZERO-SHOT INFERENCE...


  0%|          | 0/3113 [00:00<?, ?it/s]


🏆 BASELINE RESULTS (Zero-shot) - LEVEL: TEST_ALL
CER            : 0.5972
WER            : 0.8219
Exact Match    : 0.1388
BLEU           : 0.0198
  [3/3] EXPORTING EVALUATION RESULTS...
    - Detailed error analysis output path: /kaggle/working/baseline_error_analysis_test_all.csv
    - Successfully saved 3113 prediction records.

**************************************************
📊 SUMMARY OF ALL EVALUATIONS (Zero-shot VGG19)
**************************************************
--- Split: VAL_ALL ---
  CER         : 0.5586
  WER         : 0.7805
  Exact Match : 0.1984
  BLEU        : 0.0215
--- Split: TEST_LINE ---
  CER         : 0.2688
  WER         : 0.6051
  Exact Match : 0.0000
  BLEU        : 0.3052
--- Split: TEST_WORD ---
  CER         : 0.5017
  WER         : 0.8622
  Exact Match : 0.1499
  BLEU        : 0.0000
--- Split: TEST_PARAGRAPH ---
  CER         : 0.9917
  WER         : 1.0000
  Exact Match : 0.0000
  BLEU        : 0.0044
--- Split: VAL_WORD ---
  CER         : 0.4232
